# Chatbot inmobiliario RAG con OpenAI

## 1. Setup técnico: dependencias y librerías 

En este bloque instalamos las librerías necesarias e importamos dependencias.

Usaremos:
- `chromadb` para la base vectorial
- `openai` para embeddings y generación
- `pandas` y `numpy` para preparar los datos

In [1]:
%pip install chromadb openai pandas numpy python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os
import re
import unicodedata
from typing import List, Dict, Tuple
from dotenv import load_dotenv

import chromadb # base de datos vectorial
from chromadb.utils import embedding_functions # función de embedding
import numpy as np
import pandas as pd
from openai import OpenAI

load_dotenv()

MODELO_LLM = "gpt-4o-mini"
MODELO_EMBEDDING = "text-embedding-3-small"

BASE_DIR = Path(".")

PISOS_PATH = BASE_DIR / "pisos.csv"
FAQS_PATH = BASE_DIR / "faqs.txt"
OPERACIONES_PATH = BASE_DIR / "operaciones_inmobiliarias.txt"
BARRIOS_PATH = BASE_DIR / "barrios.txt"

API_KEY = os.getenv("OPENAI_API_KEY")
cliente_openai = OpenAI(api_key=API_KEY)

# la función openai_ef es el "puente" entre ChromaDB y OpenAI.
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=API_KEY,
    model_name=MODELO_EMBEDDING
)

SEED = 42
np.random.seed(SEED)

print(f"✅ Sistema inicializado — Modelo LLM: {MODELO_LLM}")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 2. Chunking: adaptar la división de documentos al dominio inmobiliario

Aquí no vamos a usar un chunking genérico. Vamos a adaptar la lógica al dominio:

- Cada fila de `pisos.csv` será un chunk independiente
- `operaciones_inmobiliarias.txt` se dividirá por secciones temáticas
- `barrrios.txt` se dividirá por bloques descriptivos de barrio
- `faqs.txt` se divide por bloques separados por líneas en blanco (pregunta + respuesta)

La idea es que cada chunk tenga sentido completo para que el retrieval sea más preciso.

In [ ]:
# normalización básica para facilitar comparaciones y búsquedas posteriores.
def normalizar_texto(texto: str) -> str:
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

# cargamos los datos desde los archivos, aplicando un fillna para evitar valores nulos
def cargar_datos() -> Tuple[pd.DataFrame, str, str, str]:
    df_pisos = pd.read_csv(PISOS_PATH).fillna('')
    
    texto_operaciones = OPERACIONES_PATH.read_text(encoding='utf-8').strip()
    texto_barrios = BARRIOS_PATH.read_text(encoding='utf-8').strip()
    texto_faqs = ''

    if FAQS_PATH.exists():
        texto_faqs = FAQS_PATH.read_text(encoding='utf-8').strip()
    return (
        df_pisos,
        texto_operaciones,
        texto_barrios,
        texto_faqs
    )

df_pisos, texto_operaciones, texto_barrios, texto_faqs = cargar_datos()

# extraemos las zonas únicas del CSV para validación y normalización.
ZONAS_CONOCIDAS = sorted(df_pisos['zona'].astype(str).unique().tolist())

# normalizamos las zonas para facilitar comparaciones posteriores
ZONAS_NORMALIZADAS = {normalizar_texto(zona): zona for zona in ZONAS_CONOCIDAS}

# formateamos cada fila del CSV en un chunk de texto con su metadata asociada.
def formatear_chunk_piso(
    fila: pd.Series
) -> Tuple[str, Dict]:
    texto = (
        f"ID: {fila['id']} | "
        f"Título: {fila['titulo']} | "
        f"Zona: {fila['zona']} | "
        f"Precio: {fila['precio']} EUR | "
        f"m2: {fila['m2']} | "
        f"Habitaciones: {fila['habitaciones']} | "
        f"Baños: {fila['banos']} | "
        f"Tipo: {fila['tipo']} | "
        f"Estado: {fila['estado']} | "
        f"Extras: {fila['extras']} | "
        f"Disponibilidad: {fila['disponibilidad']} | "
        f"Descripción: {fila['descripcion']}"
    )
    metadata = {
        'categoria': 'inmueble',
        'fuente': 'pisos.csv',
        'id_inmueble': (fila['id']),
        'zona': str(fila['zona']),
        'precio': float(fila['precio']),
        'habitaciones': float(fila['habitaciones']),
        'tipo': str(fila['tipo']),
        'estado': str(fila['estado']),
    }
    return texto, metadata

# Dividir por secciones numeradas (1., 2., 3., etc.)
def chunkear_operaciones(
    texto: str
) -> List[Tuple[str, Dict]]:
    if not texto:
        return []
    secciones = re.split(r'(?=\n\d+\.)', texto)
    salida = []
    for idx, seccion in enumerate(secciones, start=1):
        seccion = seccion.strip()
        if not seccion:
            continue
        salida.append(
            (
                seccion,
                {
                    'categoria': 'proceso_compra',
                    'fuente': 'operaciones_inmobiliarias.txt',
                    'seccion': idx,
                    'chunk_idx': idx
                }
            )
        )
    return salida

# 1 Faq = 1 Chunk
def chunkear_faqs(
    texto: str
) -> List[Tuple[str, Dict]]:
    if not texto:
        return []
    bloques = re.split(r'\n\s*\n', texto)
    salida = []
    for idx, bloque in enumerate(bloques, start=1):
        bloque = bloque.strip()
        if not bloque:
            continue
        salida.append(
            (
                bloque,
                {
                    'categoria': 'faq',
                    'fuente': 'faqs.txt',
                    'chunk_idx': idx
                }
            )
        )
    return salida

# 1 Barrio = 1 Chunk
def chunkear_barrios(
    texto: str
) -> List[Tuple[str, Dict]]:
    bloques = re.split(r'\n\s*\n', texto)
    salida = []
    for idx, bloque in enumerate(bloques, start=1):
        bloque = bloque.strip()
        if not bloque:
            continue
        primera_linea = bloque.split(':')[0].strip()
        salida.append(
            (
                bloque,
                {
                    'categoria': 'barrio',
                    'fuente': 'barrios.txt',
                    'chunk_idx': idx,
                    'zona': primera_linea
                }
            )
        )
    return salida

chunks_pisos = [
    formatear_chunk_piso(fila)
    for _, fila in df_pisos.iterrows()
]

chunks_operaciones = chunkear_operaciones(texto_operaciones)
chunks_barrios = chunkear_barrios(texto_barrios)
chunks_faqs = chunkear_faqs(texto_faqs)

print(f'Pisos: {len(chunks_pisos)} chunks')
print(f'Operaciones: {len(chunks_operaciones)} chunks')
print(f'Barrios: {len(chunks_barrios)} chunks')
print(f'FAQs: {len(chunks_faqs)} chunks')
print(ZONAS_NORMALIZADAS)

print('\nEjemplo piso:')
print(chunks_pisos[0][0])

print('\nEjemplo operación (sección completa):')
print(chunks_operaciones[0][0])

print('\nEjemplo barrio:')
print(chunks_barrios[0][0])

print('\nEjemplo FAQ:')
print(chunks_faqs[1][0])

Pisos: 25 chunks
Operaciones: 14 chunks
Barrios: 8 chunks
FAQs: 37 chunks
{'chamartin': 'Chamartín', 'chamberi': 'Chamberí', 'lavapies': 'Lavapiés', 'malasana': 'Malasaña', 'moncloa': 'Moncloa', 'retiro': 'Retiro', 'salamanca': 'Salamanca', 'vallecas': 'Vallecas'}

Ejemplo piso:
ID: 1 | Título: Ático moderno con terraza en Salamanca | Zona: Salamanca | Precio: 1180000 EUR | m2: 185 | Habitaciones: 4 | Baños: 3 | Tipo: ático | Estado: reformado | Extras: terraza|garaje|ascensor|trastero | Disponibilidad: disponible | Descripción: Ático exclusivo reformado recientemente con gran terraza y vistas despejadas. Muy luminoso durante todo el día, ideal para familias que buscan amplitud y tranquilidad en una de las mejores zonas de Madrid. Excelente conexión transporte y cerca de colegios internacionales y zonas comerciales.

Ejemplo operación (sección completa):
1. BÚSQUEDA DEL INMUEBLE

El comprador define sus necesidades y presupuesto antes de iniciar la búsqueda.

Aspectos habituales:
- ubi

## 3. Base vectorial y retrieval: ChromaDB para buscar similitud

Ya tenemos chunks. Ahora los guardamos en una base vectorial y recuperamos los más parecidos a la consulta del usuario.

La búsqueda semántica será el núcleo del retrieval. Aquí lo montamos en memoria, como en el notebook del profe, para que sea una demo limpia y fácil de seguir.

In [ ]:
cliente_chroma = chromadb.Client()

NOMBRE_COLECCION = "rag_inmobiliario"

try:
    cliente_chroma.delete_collection(
        name=NOMBRE_COLECCION
    )

except:
    pass
coleccion = cliente_chroma.create_collection(
    name=NOMBRE_COLECCION,
    embedding_function=openai_ef
)

# unificamos los chunks de las cuatro fuentes en una sola lista
todos_los_chunks = (
    chunks_pisos
    + chunks_operaciones
    + chunks_barrios
    + chunks_faqs
)

print(f"Total chunks indexados: {len(todos_los_chunks)}")
documentos = []
metadatas = []
ids = []

for idx, (texto, metadata) in enumerate(
    todos_los_chunks
):

    documentos.append(texto)
    metadatas.append(metadata)
    ids.append(f"chunk_{idx}")

# envía los textos a OpenAI (via openai_ef),
coleccion.add(
    documents=documentos,
    metadatas=metadatas,
    ids=ids
)

print("\n✅ Base vectorial creada")

# implementa el retrieval semántico
def buscar_contexto(
    consulta: str,
    top_k: int = 5
):
    resultados = coleccion.query(
        query_texts=[consulta],
        n_results=top_k
    )

    return resultados['documents'][0]

consulta_prueba = "¿Quiénes sois?"

resultados = buscar_contexto(
    consulta=consulta_prueba,
    top_k=3
)

print(f"\nConsulta de prueba: "f"{consulta_prueba}")

print("\nRESULTADOS RETRIEVAL\n")

for idx, documento in enumerate(
    resultados,
    start=1
):
    print(f"--- Resultado {idx} ---\n")
    print(documento[:500])

Total chunks indexados: 84

✅ Base vectorial creada

Consulta de prueba: Busco piso en moncloa con 2 habitaciones y balcón

RESULTADOS RETRIEVAL

--- Resultado 1 ---

ID: 19 | Título: Piso con balcón en Chamberí | Zona: Chamberí | Precio: 410000 EUR | m2: 74 | Habitaciones: 2 | Baños: 1 | Tipo: piso | Estado: buen estado | Extras: balcón|ascensor | Disponibilidad: disponible | Descripción: Piso acogedor con balcón exterior y excelente distribución. Ideal para teletrabajo y vida urbana, ubicado cerca de metro y cafeterías de moda.
--- Resultado 2 ---

ID: 12 | Título: Dúplex tranquilo en Moncloa | Zona: Moncloa | Precio: 710000 EUR | m2: 128 | Habitaciones: 3 | Baños: 3 | Tipo: dúplex | Estado: buen estado | Extras: terraza|garaje | Disponibilidad: disponible | Descripción: Dúplex espacioso en zona tranquila, ideal para familias y profesionales que trabajan desde casa. Cuenta con terraza privada y excelente acceso a universidades y transporte público.
--- Resultado 3 ---

ID: 2 | Título

In [ ]:
# ejemplo de retrieval semántico puro

consulta = (
    'Busco un piso reformado de 3 habitaciones en Chamberí'
)

# Generamos el embedding de la consulta directamente para inspeccionar sus dimensiones y valores
embedding_consulta = openai_ef([consulta])[0]

print(f'\nConsulta: {consulta}')
print(f'\nDimensiones embedding: {len(embedding_consulta)}')
print(f'Primeros valores embedding: {embedding_consulta[:10]}')

# query directa a ChromaDB
resultados = coleccion.query(query_texts=[consulta],n_results=3)

documentos = resultados["documents"][0]
distancias = resultados["distances"][0]

# mostrar resultados
for idx, (texto, distancia) in enumerate(
    zip(documentos, distancias),
    start=1
):
    print(f"\n[Resultado {idx}] "f"distancia={distancia:.4f}")
    print(texto)


Consulta: Busco un piso reformado de 3 habitaciones en Chamberí

Dimensiones embedding: 1536
Primeros valores embedding: [ 0.00602722  0.00126839 -0.00492096  0.00614548 -0.05133057  0.0249176
 -0.00021458  0.03125     0.00025177 -0.02548218]

[Resultado 1] distancia=0.2796
ID: 9 | Título: Piso reformado en Chamberí | Zona: Chamberí | Precio: 540000 EUR | m2: 92 | Habitaciones: 3 | Baños: 2 | Tipo: piso | Estado: reformado | Extras: ascensor|balcón | Disponibilidad: disponible | Descripción: Piso reformado recientemente con acabados modernos y excelente distribución. Muy luminoso y perfecto para teletrabajo gracias a sus espacios independientes. Cerca de metro y restaurantes.

[Resultado 2] distancia=0.3410
ID: 19 | Título: Piso con balcón en Chamberí | Zona: Chamberí | Precio: 410000 EUR | m2: 74 | Habitaciones: 2 | Baños: 1 | Tipo: piso | Estado: buen estado | Extras: balcón|ascensor | Disponibilidad: disponible | Descripción: Piso acogedor con balcón exterior y excelente distribuci

## 6. Generación con LLM: prompts anti-alucinación y GPT-4o-mini

El prompt debe forzar una regla simple: responder solo con lo recuperado. Si falta información, el sistema lo dice.

Eso evita respuestas inventadas y mantiene el chatbot dentro del dominio inmobiliario.

In [ ]:
def generar_respuesta(
    consulta: str,
    chunks_contexto: List[str]
) -> str:

    # unimos el contexto recuperado
    contexto = (
        "\n\n---\n\n"
        .join(chunks_contexto)
    )

    # evitar contexto vacío
    if not contexto.strip():

        return (
            "No he encontrado información "
            "suficiente para responder."
        )

    # prompt RAG
    prompt = f"""
Eres un asistente inmobiliario.

Debes responder usando SOLO
la información proporcionada.

INFORMACIÓN DISPONIBLE:
{contexto}

PREGUNTA:
{consulta}

REGLAS:
- Usa solo el contexto
- No inventes información
- Si falta información, dilo claramente
- Sé breve y claro
"""

    # llamada OpenAI
    respuesta = (
        cliente_openai
        .chat.completions.create(
            model=MODELO_LLM,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )
    )

    return (
        respuesta
        .choices[0]
        .message.content
        .strip()
    )

In [ ]:
consulta_prueba = (
    'Busco una casa cerca del centro con emphasis en cultura y ocio. Que opciones tengo?'
)

print(f'\nConsulta: {consulta_prueba}')

# =========================================================
# RETRIEVAL SEMÁNTICO
# =========================================================

resultados = buscar_contexto(
    consulta=consulta_prueba,
    top_k=3
)

# Ya son textos directamente
chunks_contexto = resultados

# =========================================================
# CONTEXTO RECUPERADO
# =========================================================

print('\nCONTEXTO RECUPERADO')

for idx, chunk in enumerate(
    chunks_contexto,
    start=1
):

    print(f'\n[Chunk {idx}]')

    print(chunk[:400])

# =========================================================
# GENERACIÓN RESPUESTA
# =========================================================

respuesta = generar_respuesta(
    consulta=consulta_prueba,
    chunks_contexto=chunks_contexto
)

# =========================================================
# RESPUESTA FINAL
# =========================================================

print('-' * 50)

print('\n🤖 RESPUESTA FINAL\n')

print(respuesta)


Consulta: Busco una casa cerca del centro con emphasis en cultura y ocio. Que opciones tengo?



CONTEXTO RECUPERADO

[Chunk 1]
Lavapiés: Zona multicultural y artística con fuerte identidad urbana y ambiente internacional. Muy valorada por perfiles creativos, estudiantes, inversores y personas atraídas por la diversidad cultural. Bien conectada por metro y Cercanías, cuenta con mercados tradicionales, espacios culturales, restaurantes internacionales y vida social intensa. Combina edificios históricos con proyectos de reha

[Chunk 2]
Malasaña: Barrio céntrico con ambiente joven, alternativo y creativo, famoso por su vida cultural, cafeterías, ocio nocturno y comercios independientes. Muy popular entre estudiantes, artistas, emprendedores digitales y profesionales jóvenes. Excelente conexión mediante metro y cercanía a múltiples zonas del centro. Ofrece una gran variedad de bares, restaurantes, coworkings y actividad social con

[Chunk 3]
Chamberí: Zona residencial con carácter clásico madrileño, muy valorada por su equilibrio entre tranquilidad y vida urbana. Atrae a jóvenes prof

## 7. Pipeline completo en una clase: `SistemaRAGInmobiliario`

Ahora encapsulamos todo en una sola clase para que el notebook quede limpio y reutilizable.

La clase se encarga de:
- Cargar e indexar datos
- Detectar filtros
- Recuperar contexto
- Generar la respuesta final

In [ ]:
class SistemaRAGInmobiliario:

    def __init__(self):
        load_dotenv()
        self.API_KEY = os.getenv("OPENAI_API_KEY")

        if not self.API_KEY:
            raise ValueError(
                "No se encontró OPENAI_API_KEY"
            )

        self.MODELO_LLM = "gpt-4o-mini"
        self.MODELO_EMBEDDING = ("text-embedding-3-small")
        
        self.BASE_DIR = Path(".")
        self.PISOS_PATH = (self.BASE_DIR / "pisos.csv")
        self.FAQS_PATH = (self.BASE_DIR / "faqs.txt")
        self.OPERACIONES_PATH = (self.BASE_DIR /"operaciones_inmobiliarias.txt")
        self.BARRIOS_PATH = (self.BASE_DIR / "barrios.txt")

        self.cliente_llm = OpenAI(api_key=self.API_KEY)

        self.openai_ef = (
            embedding_functions
            .OpenAIEmbeddingFunction(
                api_key=self.API_KEY,
                model_name=self.MODELO_EMBEDDING
            )
        )

        np.random.seed(42)

        self.cargar_datos()
        self.crear_chunks()
        self.crear_base_vectorial()

    def normalizar_texto(
        self,
        texto: str
    ) -> str:

        texto = texto.lower().strip()
        texto = unicodedata.normalize(
            "NFD",
            texto
        )

        texto = ''.join(
            c for c in texto
            if unicodedata.category(c) != 'Mn'
        )

        return texto

    def cargar_datos(self):

        self.df_pisos = pd.read_csv(
            self.PISOS_PATH
        ).fillna('')

        self.texto_operaciones = (
            self.OPERACIONES_PATH
            .read_text(encoding='utf-8')
            .strip()
        )

        self.texto_barrios = (
            self.BARRIOS_PATH
            .read_text(encoding='utf-8')
            .strip()
        )

        self.texto_faqs = ''

        if self.FAQS_PATH.exists():

            self.texto_faqs = (
                self.FAQS_PATH
                .read_text(encoding='utf-8')
                .strip()
            )

        print("✅ Datos cargados")

    def formatear_chunk_piso(
        self,
        fila: pd.Series
    ) -> Tuple[str, Dict]:

        texto = (
            f"ID: {fila['id']} | "
            f"Título: {fila['titulo']} | "
            f"Zona: {fila['zona']} | "
            f"Precio: {fila['precio']} EUR | "
            f"m2: {fila['m2']} | "
            f"Habitaciones: {fila['habitaciones']} | "
            f"Baños: {fila['banos']} | "
            f"Tipo: {fila['tipo']} | "
            f"Estado: {fila['estado']} | "
            f"Extras: {fila['extras']} | "
            f"Disponibilidad: "
            f"{fila['disponibilidad']} | "
            f"Descripción: "
            f"{fila['descripcion']}"
        )

        metadata = {
            'categoria': 'inmueble',
            'fuente': 'pisos.csv',
            'id_inmueble': int(fila['id']),
            'zona': str(fila['zona']),
            'precio': int(float(fila['precio'])),
            'habitaciones': int(
                float(fila['habitaciones'])
            ),
            'tipo': str(fila['tipo']),
            'estado': str(fila['estado'])
        }

        return texto, metadata

    def chunkear_operaciones(
        self,
        texto: str
    ) -> List[Tuple[str, Dict]]:

        if not texto:
            return []

        secciones = re.split(
            r'(?=\n\d+\.)',
            texto
        )

        salida = []

        for idx, seccion in enumerate(
            secciones,
            start=1
        ):

            seccion = seccion.strip()

            if not seccion:
                continue

            salida.append(
                (
                    seccion,
                    {
                        'categoria': (
                            'proceso_compra'
                        ),
                        'fuente': (
                            'operaciones_'
                            'inmobiliarias.txt'
                        ),
                        'chunk_idx': idx
                    }
                )
            )

        return salida

    def chunkear_faqs(
        self,
        texto: str
    ) -> List[Tuple[str, Dict]]:

        if not texto:
            return []

        bloques = re.split(
            r'\n\s*\n',
            texto
        )

        salida = []

        for idx, bloque in enumerate(
            bloques,
            start=1
        ):

            bloque = bloque.strip()

            if not bloque:
                continue

            salida.append(
                (
                    bloque,
                    {
                        'categoria': 'faq',
                        'fuente': 'faqs.txt',
                        'chunk_idx': idx
                    }
                )
            )

        return salida
    
    def chunkear_barrios(
        self,
        texto: str
    ) -> List[Tuple[str, Dict]]:

        bloques = re.split(
            r'\n\s*\n',
            texto
        )

        salida = []

        for idx, bloque in enumerate(
            bloques,
            start=1
        ):

            bloque = bloque.strip()

            if not bloque:
                continue

            primera_linea = (
                bloque
                .split(':')[0]
                .strip()
            )

            salida.append(
                (
                    bloque,
                    {
                        'categoria': 'barrio',
                        'fuente': 'barrios.txt',
                        'chunk_idx': idx,
                        'zona': primera_linea
                    }
                )
            )

        return salida

    def crear_chunks(self):

        self.chunks_pisos = [
            self.formatear_chunk_piso(fila)
            for _, fila in (
                self.df_pisos.iterrows()
            )
        ]

        self.chunks_operaciones = (
            self.chunkear_operaciones(
                self.texto_operaciones
            )
        )

        self.chunks_barrios = (
            self.chunkear_barrios(
                self.texto_barrios
            )
        )

        self.chunks_faqs = (
            self.chunkear_faqs(
                self.texto_faqs
            )
        )

        print("✅ Chunks generados")

    def crear_base_vectorial(self):

        self.cliente_chroma = (
            chromadb.Client()
        )

        nombre_coleccion = (
            "rag_inmobiliario"
        )

        try:
            self.cliente_chroma.delete_collection(
                name=nombre_coleccion
            )
        except:
            pass

        self.coleccion = (
            self.cliente_chroma
            .create_collection(
                name=nombre_coleccion,
                embedding_function=(
                    self.openai_ef
                )
            )
        )

        todos_los_chunks = (
            self.chunks_pisos
            + self.chunks_operaciones
            + self.chunks_barrios
            + self.chunks_faqs
        )

        documentos = []
        metadatas = []
        ids = []

        for idx, (
            texto,
            metadata
        ) in enumerate(todos_los_chunks):

            documentos.append(texto)
            metadatas.append(metadata)
            ids.append(f"chunk_{idx}")

        self.coleccion.add(
            documents=documentos,
            metadatas=metadatas,
            ids=ids
        )

        print(
            "✅ Base vectorial creada"
        )

    def buscar_contexto(
        self,
        consulta: str,
        top_k: int = 3
    ) -> List[Dict]:

        resultados = self.coleccion.query(
            query_texts=[consulta],
            n_results=top_k
        )

        salida = []

        documentos = resultados['documents'][0]
        metadatas = resultados['metadatas'][0]
        distancias = resultados['distances'][0]

        for texto, metadata, distancia in zip(
            documentos,
            metadatas,
            distancias
        ):

            salida.append({
                'texto': texto,
                'metadata': metadata,
                'distancia': distancia
            })

        return salida

    def construir_contexto(
        self,
        resultados: List[Dict]
    ) -> str:

        if not resultados:
            return ''

        bloques = []

        for resultado in resultados:
            metadata = resultado.get('metadata',{})
            cabecera = (f"[Fuente: "f"{metadata.get('fuente')} "f"| Categoría: "f"{metadata.get('categoria')}]")
            bloques.append(f"{cabecera}\n"f"{resultado['texto']}")

        return (
            '\n\n---\n\n'
            .join(bloques)
        )

    def generar_respuesta(
        self,
        consulta: str,
        contexto: str
    ) -> str:

        if not contexto.strip():

            return (
                'No he encontrado '
                'información suficiente.'
            )

        contexto = contexto[:12000]

        mensaje_sistema = """
Eres un asistente inmobiliario preciso y prudente.

Tu tarea es responder únicamente usando
la información presente en el contexto.

Reglas:
- No inventes información
- No completes datos faltantes
- Si algo no aparece en el contexto, dilo
- Prioriza pisos cuando el usuario busque inmuebles
- Responde de forma clara y breve
"""

        mensaje_usuario = f"""
CONSULTA:
{consulta}

CONTEXTO:
{contexto}
"""

        respuesta = (
            self.cliente_llm
            .chat.completions.create(
                model=self.MODELO_LLM,
                messages=[
                    {
                        'role': 'system',
                        'content': mensaje_sistema
                    },
                    {
                        'role': 'user',
                        'content': mensaje_usuario
                    }
                ],
                temperature=0.2
            )
        )

        return (
            respuesta
            .choices[0]
            .message.content
            .strip()
        )

    def preguntar(
        self,
        consulta: str,
        top_k: int = 3
    ):

        print(
            f"\n🔎 CONSULTA: "
            f"{consulta}"
        )

        resultados = self.buscar_contexto(
            consulta=consulta,
            top_k=top_k
        )

        print(
            "\n📚 RESULTADOS RECUPERADOS"
        )

        for idx, resultado in enumerate(
            resultados,
            start=1
        ):

            metadata = resultado[
                'metadata'
            ]

            print(
                f"\n[Resultado {idx}] "
                f"distancia="
                f"{resultado['distancia']:.4f}"
            )

            if 'zona' in metadata:

                print(
                    f"📍 Zona: "
                    f"{metadata['zona']}"
                )

            print(
                "\n📄 Texto:\n"
            )

            print(
                resultado['texto'][:400]
            )

        contexto = self.construir_contexto(
            resultados
        )

        respuesta = self.generar_respuesta(
            consulta=consulta,
            contexto=contexto
        )

        print(
            "\nRESPUESTA FINAL\n"
        )

        print(respuesta)

sistema = SistemaRAGInmobiliario()

sistema.preguntar('Busco un piso reformado de 3 habitaciones en malasana por menos de 600.000 euros')


✅ Datos cargados
✅ Chunks generados
✅ Base vectorial creada

🔎 CONSULTA: Busco un piso reformado de 3 habitaciones en malasana por menos de 600.000 euros

📚 RESULTADOS RECUPERADOS

[Resultado 1] distancia=0.3881
📍 Zona: Malasaña

📄 Texto:

ID: 5 | Título: Dúplex luminoso en Malasaña | Zona: Malasaña | Precio: 620000 EUR | m2: 110 | Habitaciones: 3 | Baños: 2 | Tipo: dúplex | Estado: reformado | Extras: balcón|ascensor | Disponibilidad: disponible | Descripción: Dúplex con diseño moderno y espacios abiertos, ideal para parejas jóvenes que buscan vivir en una zona céntrica y dinámica. Muy luminoso, reformado recientemente y cerca de m

[Resultado 2] distancia=0.4032
📍 Zona: Vallecas

📄 Texto:

ID: 16 | Título: Piso económico reformado en Vallecas | Zona: Vallecas | Precio: 225000 EUR | m2: 60 | Habitaciones: 2 | Baños: 1 | Tipo: piso | Estado: reformado | Extras: balcón | Disponibilidad: disponible | Descripción: Piso reformado recientemente con excelente relación calidad-precio. Ideal p